# SIF Lag Analysis — Iowa 2023

Tests whether SIF from 6–12 weeks ago predicts current drought conditions,
following the approach of Parazoo et al. (2024).

**Prerequisites:** Run `02_sif_eda.ipynb` first to generate the SIF file inventory.
Converted from `scripts/analysis/lag_analysis_optimized.py`.

In [ ]:
import sys
import os
import gc
from pathlib import Path
import numpy as np
import xarray as xr
import rioxarray as rxr
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats

# Auto-detect HPC vs local for backend
_is_hpc = os.environ.get('SLURM_JOB_ID') or os.environ.get('PBS_JOBID')
matplotlib.use('Agg' if _is_hpc else 'TkAgg')

_root_env    = os.environ.get('SIF_ROOT')
project_root = Path(_root_env) if _root_env else Path('../../..').resolve()

sif_dir     = project_root / 'data' / 'raw' / 'sif'
drought_dir = project_root / 'data' / 'raw' / 'drought_usdm'
figures_dir = project_root / 'figures' / 'lag_analysis'
figures_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print(f'SIF dir:      {sif_dir}')

## 1. SIF File Inventory

In [ ]:
file_info = []
for fpath in sorted(sif_dir.glob('sif_ann_*.nc4')):
    stem  = fpath.stem
    parts = stem.split('_')
    yyyymm = parts[2][:6]
    half   = parts[2][6]
    file_info.append({
        'path'  : fpath,
        'year'  : yyyymm[:4],
        'month' : yyyymm[4:6],
        'half'  : half,
        'yyyymm': yyyymm,
    })

YEAR_TARGET          = '2023'
file_info_2023       = [f for f in file_info if f['year'] == YEAR_TARGET]
file_info_2023_sorted = sorted(file_info_2023,
                                key=lambda x: (x['year'], x['month'], x['half']))

print(f'Total SIF files: {len(file_info)}')
print(f'2023 SIF files:  {len(file_info_2023_sorted)}')

## 2. Iowa Bounds and Helper Functions

In [ ]:
iowa_bounds = (-96.64, 40.38, -90.14, 43.50)  # west, south, east, north

def load_sif_da(info):
    ds = xr.open_dataset(info['path'])
    return ds['sif_ann']

def _iowa_spatial(da, info):
    lat_dim = 'latitude' if 'latitude' in da.dims else 'lat'
    lon_dim = 'longitude' if 'longitude' in da.dims else 'lon'
    return da.sel(**{
        lat_dim: slice(iowa_bounds[1] - 0.5, iowa_bounds[3] + 0.5),
        lon_dim: slice(iowa_bounds[0] - 0.5, iowa_bounds[2] + 0.5),
    })

def compute_sif_climatology(month_int):
    month_str   = f'{month_int:02d}'
    month_files = [f for f in file_info if f['month'] == month_str]
    arrays = []
    for info in month_files:
        da = _iowa_spatial(load_sif_da(info), info)
        arrays.append(da.values.astype(np.float32))
        da.close()
    if not arrays:
        return None, None
    stack = np.stack(arrays, axis=0)
    return np.nanmean(stack, axis=0), np.nanstd(stack, axis=0)

def load_drought_for_period(drought_info):
    month    = int(drought_info['month'])
    half_idx = 1 if drought_info['half'] == 'a' else 2
    tif = drought_dir / f"USDM_Iowa_DM_{drought_info['year']}-{month:02d}_{half_idx}.tif"
    if not tif.exists():
        return None
    da = rxr.open_rasterio(tif).squeeze(drop=True)
    if da.rio.nodata is not None:
        da = da.where(da != da.rio.nodata)
    if da.rio.crs is None:
        da = da.rio.write_crs('EPSG:4326')
    return da

print('Helper functions defined.')

## 3. Lag Analysis

For each lag (6, 8, 10, 12 weeks), pairs SIF from N half-months ago with the current
drought category. Tests whether vegetation stress (low SIF anomaly) leads the USDM
drought signal. Following Parazoo et al. (2024).

**Lags:** 6 weeks = 3 half-months, 8 = 4, 10 = 5, 12 = 6

In [ ]:
lags_halfmonths = {6: 3, 8: 4, 10: 5, 12: 6}

def get_lagged_pairs(lag_halfmonths):
    sif_vals, drought_vals = [], []
    for i, drought_info in enumerate(file_info_2023_sorted):
        if i < lag_halfmonths:
            continue
        sif_info = file_info_2023_sorted[i - lag_halfmonths]
        month_sif = int(sif_info['month'])

        clim_mean, clim_std = compute_sif_climatology(month_sif)
        if clim_mean is None:
            continue

        sif_da  = _iowa_spatial(load_sif_da(sif_info), sif_info)
        sif_arr = sif_da.values.astype(np.float32)

        # Trim climatology to match SIF spatial subset shape
        h, w = sif_arr.shape
        cm   = clim_mean[:h, :w] if clim_mean.ndim == 2 else clim_mean
        cs   = clim_std[:h,  :w] if clim_std.ndim  == 2 else clim_std
        sif_z = (sif_arr - cm) / (cs + 1e-6)

        drought_da = load_drought_for_period(drought_info)
        if drought_da is None:
            del sif_da, sif_arr, sif_z, clim_mean, clim_std
            gc.collect()
            continue

        d_arr = drought_da.values.astype(np.float32)
        # Match spatial shapes (coarser drought to SIF grid — use center crop)
        min_h = min(sif_z.shape[0], d_arr.shape[0])
        min_w = min(sif_z.shape[1], d_arr.shape[1])
        s = sif_z[:min_h, :min_w]
        d = d_arr[:min_h, :min_w]

        valid = np.isfinite(s) & np.isfinite(d) & (d >= 0) & (d <= 5)
        if valid.any():
            sif_vals.append(s[valid].copy())
            drought_vals.append(d[valid].copy())

        del sif_da, sif_arr, sif_z, clim_mean, clim_std, drought_da, d_arr
        if i % 5 == 0:
            gc.collect()

    if not sif_vals:
        return None, None
    return np.concatenate(sif_vals), np.concatenate(drought_vals)

print('Lag function defined. Running analysis...')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    'SIF z-score Distribution by Drought Category\n'
    'Iowa Agricultural Lands, 2023 — Time Lag Analysis',
    fontsize=13, fontweight='bold'
)

for ax, lag_weeks in zip(axes.flatten(), [6, 8, 10, 12]):
    lag_hm = lags_halfmonths[lag_weeks]
    print(f'Processing {lag_weeks}-week lag...')

    sif_all, drought_all = get_lagged_pairs(lag_hm)

    if sif_all is None:
        ax.text(0.5, 0.5, f'No data ({lag_weeks}-wk lag)',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{lag_weeks}-Week Lag')
        continue

    cats   = np.clip(np.round(drought_all).astype(int), 0, 5)
    groups = {c: sif_all[cats == c] for c in range(6) if (cats == c).any()}
    avail  = sorted(groups)

    bp = ax.boxplot([groups[c] for c in avail], labels=[str(c) for c in avail],
                   patch_artist=True, showmeans=True, widths=0.6)

    colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(avail)))
    for patch, col in zip(bp['boxes'], colors):
        patch.set_facecolor(col)
        patch.set_alpha(0.7)
    for med in bp['medians']:
        med.set_color('#1a1a1a')
        med.set_linewidth(2)

    ax.axhline(0, color='gray', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Drought Category (DM)')
    ax.set_ylabel('SIF z-score')
    ax.set_title(f'{lag_weeks}-Week Lag  (n={len(sif_all):,})')
    ax.grid(True, alpha=0.3, axis='y')

    print(f'  {lag_weeks}-wk: {len(sif_all):,} pairs, {len(avail)} categories')
    del sif_all, drought_all, groups
    gc.collect()

plt.tight_layout()
out_path = figures_dir / 'lag_analysis_2023.png'
fig.savefig(out_path, dpi=300, bbox_inches='tight')
print(f'\nSaved: {out_path}')
plt.show()